# Part 1: Find the HUC8 region for the area of interest.

In [2]:
import geopandas as gpd
import pandas as pd

################## Input Files #########################################
########################################################################
# Load AOI shapefile
aoi = gpd.read_file("/content/boundary.shp")

# Load full HUC8 shapefile (in EPSG:5070)
huc8 = gpd.read_file("/content/HUC_8_EPSG_5070.shp")
########################################################################

# Ensure CRS match
if aoi.crs != huc8.crs:
    aoi = aoi.to_crs(huc8.crs)

# Spatial join: HUC8s intersecting AOI
intersecting = gpd.sjoin(huc8, aoi, how="inner", predicate="intersects")

# Extract unique HUC8 codes and force them into 8-digit strings (with leading zeros)
unique_huc8_codes = sorted(
    intersecting["HUC8"].astype(str).str.zfill(8).unique()
)

# Save to CSV (with header, preserve leading zeros)
pd.DataFrame(unique_huc8_codes, columns=["HUC8"]).to_csv("HUC8.csv", index=False)

print("✅ Saved to HUC8.csv with 8-digit strings")


✅ Saved to HUC8.csv with 8-digit strings


# Part 2 : If you AOI lies in single HUC region follow steps 1 to 9

Step 1: Pip install FIMSERVE

In [3]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 46.9 MB/s eta 0:00:00


In [4]:
!uv pip install fimserve

Using Python 3.12.13 environment at: /usr
Resolved 266 packages in 4.77s
Prepared 64 packages in 10.70s
Uninstalled 10 packages in 224ms
Installed 65 packages in 597ms
 + aiobotocore==2.26.0
 + aioitertools==0.13.0
 + aniso8601==10.0.1
 + apache-sedona==1.8.1
 + appdirs==1.4.4
 + arch==7.2.0
 + asciitree==0.3.3
 + async-lru==2.3.0
 + awscli==1.43.5
 + baseflow==0.1.0
 + boto3==1.41.5
 + botocore==1.41.5
 + cachelib==0.13.0
 + cftime==1.6.5
 + color-operations==0.2.0
 + colorama==0.4.6
 - dask==2026.3.0
 + dask==2025.12.0
 + dataretrieval==1.1.3
 + deprecated==1.3.1
 - docutils==0.21.2
 + docutils==0.19
 + fasteners==0.20
 + fimeval==0.1.62
 + fimserve==0.1.94
 + flask-caching==2.4.0
 + flask-cors==6.0.2
 + flask-restx==1.3.2
 + geocube==0.7.1
 + jedi==0.19.2
 + jmespath==1.1.0
 + json5==0.14.0
 + jupyter==1.1.1
 + jupyter-lsp==2.3.1
 + jupyterlab==4.5.6
 + jupyterlab-server==2.28.0
 + kaleido==0.2.1
 + kerchunk==0.2.7
 + localtileserver==0.11.0
 - lxml==6.0.3
 + lxml==5.4.0
 + morecant

Step 2: Import necessary libraries

In [5]:
#Importing fimserve and other necessary libraries
import fimserve as fm
import pandas as pd
from pathlib import Path #Incase user wants to run FIM for multiple HUCs

Step 3: Enter HUC8 information

In [6]:
#Enter the value of huc from the HUC8.CSV file generated from step 1.
##################Input-----files#######################################
########################################################################
huc = "03170002"

########################################################################
#huc = pd.read_csv('HUC.csv', dtype=str)
#huc['HUC'] = huc['HUC'].str.zfill(8)  # Ensures exactly 8 characters

# Loop through each HUC8 value

#Necessary variables for downloading the NWM data. These are necessary only if you are using NWM retrospective data for generating flood maps as in Step 5 Case 1.
#start_date = "2022-04-01"
#end_date = "2022-04-03"

#Value times where user wants to generate the FIM, any number as per user requirement
#value_times = ["2022-04-02 12:00:00"]

Step 4: Download HUC8 rasters

In [7]:
#For instance, If user needs stream having order more than 4
#stream_order = '>4'     #It supports >, <, >=, <=, =, and also [3,4] range if the user is picky on specific stream order

#stream_order=[6]# Download the data for one huc
fm.DownloadHUC8(huc)
# for i in huc['HUC']:
#     fm.DownloadHUC8(i)

Repository cloned into: /content/code/inundation-mapping (version: main)
Data for HUC 03170002 downloaded to /content/output/flood_03170002/03170002
Copied /content/output/flood_03170002/03170002/branch_ids.csv to /content/output/flood_03170002/fim_inputs.csv as fim_inputs.csv.
Unique feature IDs saved to /content/output/flood_03170002/feature_IDs.csv.


Step 5: Download NWM discharge data

Case 1: If you want to run the FIMserve for a single value of discharge for each reach ID, run this code. Our workflow usually uses return period flows for multiple FIM generation.

In [ ]:
fm.getNWMretrospectivedata(huc, start_date, end_date, value_times=value_times)

NWM discharge data saved to /content/output/flood_12100201/discharge/nwm30_retrospective.
Discharge values saved to /content/data/inputs/NWM_20220402120000_12100201.csv


Case 2: If you want to run the FIMserve for multiple return period flows run this code. The return period flows includes flows for return periods 2 year, 5 year, 10 year, 25 year, 50 year, and 100 year flows.

In [8]:
import os, io, time
import pandas as pd
import requests
from google.colab import userdata

# === Inputs ===
# Assume `huc` is already defined earlier in your notebook/session, e.g.:
feature_ids_path = f"/content/output/flood_{huc}/feature_IDs.csv"   # header: feature_id (or NWM_ID/comid)
API_URL          = "https://nwm-api.ciroh.org/"
RETURN_EP        = f"{API_URL}/return-period"
try:
    API_KEY = userdata.get('CIROH_API_KEY')
    if not API_KEY:
        raise ValueError("CIROH_API_KEY secret not found")
    print("✅ API key loaded securely from Colab Secrets")
except Exception as e:
    print(f"❌ Error loading API key: {e}")
    print("\n📝 Setup Instructions:")
    print("1. Click the 🔑 key icon in the left sidebar")
    print("2. Click 'Add new secret'")
    print("3. Name: CIROH_API_KEY")
    print("4. Value: Your CIROH API key")
    print("5. Click 'Add secret'")
    print("6. Run this cell again")
    raise


# Output directory for per-RP CSVs
per_rp_out_dir = "/content/data/inputs"

# === Tunables ===
BATCH_SIZE   = 200          # 100–300 is usually safe for URL length
TIMEOUT_S    = 60
MAX_RETRIES  = 3
RETRY_SLEEP  = 2

# ---- Load & clean IDs ----
raw = pd.read_csv(feature_ids_path)
col = next((c for c in raw.columns if c.lower().strip() in ("feature_id","nwm_id","comid")), None)
if not col:
    raise ValueError("CSV must contain feature_id (or NWM_ID/comid).")
ids_series = pd.to_numeric(raw[col], errors="coerce").dropna().astype("int64")
ids_series = ids_series[ids_series > 0]
ids_unique = ids_series.drop_duplicates().tolist()
if not ids_unique:
    raise ValueError("No valid integer IDs after cleaning.")

# Preserve original order (including duplicates) for final outputs
order_df = pd.DataFrame({"feature_id": ids_series.tolist(), "_order": range(len(ids_series))})

headers = {"x-api-key": API_KEY}

def fetch_batch(id_batch):
    params = {
        "comids": ",".join(map(str, id_batch)),
        "output_format": "csv",
        "order_by_comid": True
    }
    last_err = None
    for attempt in range(1, MAX_RETRIES+1):
        try:
            r = requests.get(RETURN_EP, params=params, headers=headers, timeout=TIMEOUT_S)
            if r.status_code == 200:
                return pd.read_csv(io.StringIO(r.text))
            last_err = f"HTTP {r.status_code}: {r.text[:400]}"
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(RETRY_SLEEP * attempt)
                continue
            break
        except requests.exceptions.RequestException as e:
            last_err = f"Request error: {e}"
            time.sleep(RETRY_SLEEP * attempt)
    raise RuntimeError(f"Batch failed (first 5 IDs {id_batch[:5]} …): {last_err}")

# ---- Fetch all batches ----
frames = []
for i in range(0, len(ids_unique), BATCH_SIZE):
    batch = ids_unique[i:i+BATCH_SIZE]
    print(f"Fetching {len(batch)} IDs [{i}-{i+len(batch)-1}] …")
    dfb = fetch_batch(batch)
    if "Unnamed: 0" in dfb.columns:
        dfb = dfb.drop(columns=["Unnamed: 0"])
    frames.append(dfb)

if not frames:
    raise RuntimeError("No data returned from API.")

df = pd.concat(frames, ignore_index=True)

# Normalize column names
lower = {c.lower(): c for c in df.columns}
fid_col = lower.get("feature_id", lower.get("comid"))
if fid_col != "feature_id":
    df = df.rename(columns={fid_col: "feature_id"})

# Identify/rename return-period columns to e.g., 2_year, 5_year, …
rp_cols_raw = [c for c in df.columns if "return_period" in c.lower()]
if not rp_cols_raw:
    raise RuntimeError("No return_period columns found in API response.")

rename_map = {c: f"{c.split('_')[-1]}_year" for c in rp_cols_raw}  # keeps '2_year', '5_year', etc.
df = df.rename(columns=rename_map)

# Keep one row per feature_id from API (already ordered by comid)
df_one = df.drop_duplicates(subset=["feature_id"])

# Align to original order (keeps duplicates if present)
aligned = order_df.merge(
    df_one[["feature_id"] + list(rename_map.values())],
    on="feature_id",
    how="left"
).sort_values("_order").drop(columns=["_order"])

# ---- Write per-return-period CSVs ----
# Convert '2_year' -> '2year', '5_year' -> '5year' for filenames
os.makedirs(per_rp_out_dir, exist_ok=True)

written = []
for rp_col in sorted(rename_map.values(), key=lambda x: int(x.split("_")[0])):  # numeric sort by RP
    out_df = aligned[["feature_id", rp_col]].rename(columns={rp_col: "discharge"})
    # Filename: 2year_{huc}.csv, 5year_{huc}.csv, …
    rp_label = rp_col.replace("_year", "year")  # '2_year' -> '2year'
    out_fp = os.path.join(per_rp_out_dir, f"{rp_label}_{huc}.csv")
    out_df.to_csv(out_fp, index=False)
    written.append((rp_col, out_fp))

print("✅ Wrote per-return-period files:")
for rp_col, path in written:
    print(f"  - {rp_col}: {path}")

# ---- Diagnostics: print missing IDs message (no CSV written) ----
rp_cols = list(rename_map.values())
missing_ids = aligned[aligned[rp_cols].isna().all(axis=1)]["feature_id"].astype("int64").tolist()

if missing_ids:
    # Show up to 20 to avoid overly noisy logs
    preview = ", ".join(map(str, missing_ids[:20]))
    more = f" … (+{len(missing_ids)-20} more)" if len(missing_ids) > 20 else ""
    print(
        f"⚠️ {len(missing_ids)} ID(s) returned no return-period values: {preview}{more}\n"
        "If they lie in your area, be judicious and give relevant values; if not, don't worry."
    )
else:
    print("✅ All feature_ids received at least one return-period value.")


✅ API key loaded securely from Colab Secrets
Fetching 200 IDs [0-199] …
Fetching 200 IDs [200-399] …
Fetching 200 IDs [400-599] …
Fetching 200 IDs [600-799] …
Fetching 200 IDs [800-999] …
Fetching 200 IDs [1000-1199] …
Fetching 200 IDs [1200-1399] …
Fetching 200 IDs [1400-1599] …
Fetching 200 IDs [1600-1799] …
Fetching 144 IDs [1800-1943] …
✅ Wrote per-return-period files:
  - 2_year: /content/data/inputs/2year_03170002.csv
  - 5_year: /content/data/inputs/5year_03170002.csv
  - 10_year: /content/data/inputs/10year_03170002.csv
  - 25_year: /content/data/inputs/25year_03170002.csv
  - 50_year: /content/data/inputs/50year_03170002.csv
  - 100_year: /content/data/inputs/100year_03170002.csv
⚠️ 6 ID(s) returned no return-period values: 18179855, 18179843, 18179853, 18179851, 18179847, 18179841
If they lie in your area, be judicious and give relevant values; if not, don't worry.


Step 6: Run FIM


In [9]:
# run the FIM model, It will run the Hand model for the specified huc, if user is going through the multiple hucs, downloading discharge and everything for multiple huc , they can run it nicely for any number of HUC
fm.runOWPHANDFIM(huc, depth=True)

#!NOT RECOMMENDED, But if user wants depths using this model, mark depth = True
# fm.runOWPHANDFIM(huc, depth=True)

Completed in 0.8 minutes.

Inundation mapping for 03170002 completed successfully.
Completed in 0.69 minutes.

Inundation mapping for 03170002 completed successfully.
Completed in 0.68 minutes.

Inundation mapping for 03170002 completed successfully.
Completed in 0.86 minutes.

Inundation mapping for 03170002 completed successfully.
Completed in 0.67 minutes.

Inundation mapping for 03170002 completed successfully.
Completed in 0.65 minutes.

Inundation mapping for 03170002 completed successfully.


Step 7: Install geopandas for raster operation

In [10]:
!pip install geopandas rasterio pyproj fiona shapely

Step 8: Find the most downstream reach_ID in your domain


In [11]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

##################Input-----files#######################################
########################################################################
aoi_path = "/content/boundary.shp"  # your AOI polygon shapefile
########################################################################

net_path = f"/content/output/flood_{huc}/{huc}/nwm_subset_streams.gpkg"
feature_ids_csv = f"/content/output/flood_{huc}/feature_IDs.csv"  # optional (for membership check)
# ==========================

# Helper: pick a column name case-insensitively from several candidates
def pick(cols, *names):
    L = {c.lower(): c for c in cols}
    for n in names:
        if n.lower() in L:
            return L[n.lower()]
    return None

def yesno(v):
    if v is None:
        return "N/A"
    return "Yes" if bool(v) else "No"

# ---- Load data ----
streams = gpd.read_file(net_path)
aoi = gpd.read_file(aoi_path)
if streams.crs != aoi.crs:
    aoi = aoi.to_crs(streams.crs)

# Identify ID and downstream columns
fid_col = pick(streams.columns, "feature_id", "comid", "ID", "id", "nhdplusid")
to_col  = pick(streams.columns, "tocomid", "to", "to_comid")
assert fid_col and to_col, f"Couldn't find ID/to columns. Have: {streams.columns.tolist()}"

# Keep only what we need
streams2 = streams[[fid_col, to_col, "geometry"]].copy()

# ---- Subset to AOI and find outlet candidates ----
inside = gpd.clip(streams2, aoi).copy()
inside[fid_col] = inside[fid_col].astype(str)
inside[to_col]  = inside[to_col].astype(str)

inside_ids = set(inside[fid_col])
boundary_line = aoi.geometry.unary_union.boundary

# Reaches whose 'to' leaves the AOI are outlets
cands = inside[~inside[to_col].isin(inside_ids)].copy()
cands["touches_boundary"] = cands.geometry.intersects(boundary_line)

# Prefer those touching the boundary
primary = cands[cands["touches_boundary"]]
if primary.empty:
    primary = cands  # fallback

# Build clean report
primary = (
    primary[[fid_col, to_col]]
    .drop_duplicates()
    .rename(columns={fid_col: "last_inside", to_col: "first_outside"})
    .reset_index(drop=True)
)

# Membership check vs feature_IDs.csv (optional)
feat_set = set()
if Path(feature_ids_csv).exists():
    fdf = pd.read_csv(feature_ids_csv)
    # assume first column contains the IDs
    ref_col = fdf.columns[0]
    feat_set = set(fdf[ref_col].astype(str))

def in_feat(x):
    return (x in feat_set) if feat_set else None

primary["last_inside_in_feature_IDs"]   = primary["last_inside"].map(in_feat)
primary["first_outside_in_feature_IDs"] = primary["first_outside"].map(in_feat)

# ---- Print final output in your requested format ----
if primary.empty:
    print("No outlet candidate found.")
else:
    top = primary.iloc[0]
    print(f"last Id inside: {top['last_inside']}   Present in feature_id list: {yesno(top['last_inside_in_feature_IDs'])}")
    print(f"First ID outside: {top['first_outside']}   Present in feature id list : {yesno(top['first_outside_in_feature_IDs'])}")

# (Optional) If you also want to see all candidates in the same style, uncomment:
# for _, r in primary.iterrows():
#     print(f\"last Id inside: {r['last_inside']}   Present in feature_id list: {yesno(r['last_inside_in_feature_IDs'])}\")
#     print(f\"First ID outside: {r['first_outside']}   Present in feature id list : {yesno(r['first_outside_in_feature_IDs'])}\")
#     print()


last Id inside: 18181565   Present in feature_id list: Yes
First ID outside: 18181571   Present in feature id list : Yes


Step 9: Upload your shapefile of AOI and clip the FIM for HUC. The reach ID is used for naming the final clipped raster flood map file.

In [12]:
# 📚 Imports
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import pandas as pd

##################Input-----files#######################################
########################################################################
ReachID = "18181565"       # Replace with the most downstream ReachID (string) from step 7. Use the last ID inside first
shapefile_path = "/content/boundary.shp"  # You must upload all .shp components
#######################################################################
unit = "cms"                 # Always 'cms' for FIMServe

# 📁 Paths
raster_folder = f"/content/output/flood_{huc}/{huc}_inundation"
csv_folder = "/content/data/inputs"
output_folder = "/content/final_map"
os.makedirs(output_folder, exist_ok=True)
# 📌 Load shapefile and reproject to EPSG:5070
gdf = gpd.read_file(shapefile_path)
print(f"📌 Input shapefile CRS: {gdf.crs}")
gdf_5070 = gdf.to_crs(epsg=5070)
print("🔁 Reprojected shapefile to EPSG:5070")

# 🔄 Process each *_depth.tif raster
for raster_path in glob.glob(os.path.join(raster_folder, '*_depth.tif')):
    raster_filename = os.path.basename(raster_path)
    print(f"\n🔍 Clipping: {raster_filename}")

    try:
        # Extract base (remove _depth.tif)
        base_name = raster_filename.replace('_depth.tif', '')  # e.g., '100year_12100201'

        # Expected CSV name: e.g., '100year_12100201.csv'
        csv_filename = f"{base_name}.csv"
        csv_path = os.path.join(csv_folder, csv_filename)

        if not os.path.exists(csv_path):
            print(f"⚠️ CSV not found: {csv_filename}, skipping.")
            continue

        # Read CSV and search for ReachID
        df = pd.read_csv(csv_path)
        df = df.astype(str)
        match_row = df[df.apply(lambda row: ReachID in row.values, axis=1)]

        if match_row.empty:
            print(f"⚠️ ReachID {ReachID} not found in {csv_filename}, skipping.")
            continue

        # Extract and round discharge value
        discharge_value = int(float(match_row.iloc[0][1]))

        # Clip raster
        with rasterio.open(raster_path) as src:
            out_image, out_transform = mask(src, gdf_5070.geometry, crop=True)
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": src.crs
            })

        # Final output name WITHOUT the year prefix
        output_name = f"HAND_{ReachID}_{discharge_value}_{unit}_depth.tif"
        output_path = os.path.join(output_folder, output_name)

        # Save
        with rasterio.open(output_path, 'w', **out_meta) as dest:
            dest.write(out_image)

        print(f"✅ Saved to: {output_path}")

    except Exception as e:
        print(f"❌ Error processing {raster_filename}: {e}")


📌 Input shapefile CRS: EPSG:4326
🔁 Reprojected shapefile to EPSG:5070

🔍 Clipping: 100year_03170002_depth.tif
✅ Saved to: /content/final_map/HAND_18181565_2472_cms_depth.tif

🔍 Clipping: 50year_03170002_depth.tif
✅ Saved to: /content/final_map/HAND_18181565_2192_cms_depth.tif

🔍 Clipping: 10year_03170002_depth.tif
✅ Saved to: /content/final_map/HAND_18181565_1530_cms_depth.tif

🔍 Clipping: 2year_03170002_depth.tif
✅ Saved to: /content/final_map/HAND_18181565_775_cms_depth.tif

🔍 Clipping: 5year_03170002_depth.tif
✅ Saved to: /content/final_map/HAND_18181565_1229_cms_depth.tif

🔍 Clipping: 25year_03170002_depth.tif
✅ Saved to: /content/final_map/HAND_18181565_1910_cms_depth.tif


Step 10: Download the final files

In [13]:
# 📦 Zip the folder
!zip -r /content/final_map.zip /content/final_map > /dev/null

# 📥 Provide download link
from google.colab import files
files.download('/content/final_map.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part 3: If your AOI intersects with multiple HUCs, follow steps 10 to 18

Step 10: Pip install FIMSERVE

In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 70.8 MB/s eta 0:00:00


In [ ]:
! uv pip install fimserve

Using Python 3.12.13 environment at: /usr
Resolved 266 packages in 15.65s
Prepared 60 packages in 11.08s
Uninstalled 9 packages in 144ms
Installed 61 packages in 569ms
 + aiobotocore==2.26.0
 + aioitertools==0.13.0
 + aniso8601==10.0.1
 + apache-sedona==1.8.1
 + appdirs==1.4.4
 + arch==7.2.0
 + asciitree==0.3.3
 + async-lru==2.3.0
 + awscli==1.43.5
 + baseflow==0.1.0
 + boto3==1.41.5
 + botocore==1.41.5
 + cachelib==0.13.0
 + color-operations==0.2.0
 + colorama==0.4.6
 - dask==2026.3.0
 + dask==2025.12.0
 + dataretrieval==1.1.3
 + deprecated==1.3.1
 - docutils==0.21.2
 + docutils==0.19
 + fasteners==0.20
 + fimeval==0.1.61
 + fimserve==0.1.94
 + flask-caching==2.3.1
 + flask-cors==6.0.2
 + flask-restx==1.3.2
 + geocube==0.7.1
 + jedi==0.19.2
 + jmespath==1.1.0
 + json5==0.14.0
 + jupyter==1.1.1
 + jupyter-lsp==2.3.1
 + jupyterlab==4.5.6
 + jupyterlab-server==2.28.0
 + kaleido==0.2.1
 + kerchunk==0.2.7
 + localtileserver==0.11.0
 - lxml==6.0.2
 + lxml==5.4.0
 + morecantile==7.0.3
 + myp

Step 11: Import necessary libraries

In [ ]:
#Importing fimserve and other necessary libraries
import fimserve as fm
import pandas as pd
from pathlib import Path #Incase user wants to run FIM for multiple HUCs

Step 12: Download multipe HUCs

In [ ]:
huc = pd.read_csv('HUC8.csv', dtype=str)
#if you want for the certain stream order
#stream_order='6' or it supports '>, <, <=,>=, [3,5]
#fm.DownloadHUC8(i,stream_order)
# Loop through each HUC8 value
for i in huc['HUC8']:
    fm.DownloadHUC8(i)

Repository cloned into: /content/code/inundation-mapping (version: main)
Data for HUC 10290203 downloaded to /content/output/flood_10290203/10290203
Copied /content/output/flood_10290203/10290203/branch_ids.csv to /content/output/flood_10290203/fim_inputs.csv as fim_inputs.csv.
Unique feature IDs saved to /content/output/flood_10290203/feature_IDs.csv.
Repository already exists at /content/code/inundation-mapping. Skipping clone.
Data for HUC 10300102 downloaded to /content/output/flood_10300102/10300102
Copied /content/output/flood_10300102/10300102/branch_ids.csv to /content/output/flood_10300102/fim_inputs.csv as fim_inputs.csv.
Unique feature IDs saved to /content/output/flood_10300102/feature_IDs.csv.
Repository already exists at /content/code/inundation-mapping. Skipping clone.
Data for HUC 10300200 downloaded to /content/output/flood_10300200/10300200
Copied /content/output/flood_10300200/10300200/branch_ids.csv to /content/output/flood_10300200/fim_inputs.csv as fim_inputs.csv.

Step 13: Download return period flow data for each feature IDs in each HUC.

In [ ]:
import os, io, time
import pandas as pd
import requests
from google.colab import userdata

# === Secure API Key Loading ===
try:
    API_KEY = userdata.get('CIROH_API_KEY')
    if not API_KEY:
        raise ValueError("CIROH_API_KEY secret not found")
    print("✅ API key loaded securely from Colab Secrets")
except Exception as e:
    print(f"❌ Error loading API key: {e}")
    print("\n📝 Setup Instructions:")
    print("1. Click the 🔑 key icon in the left sidebar")
    print("2. Click 'Add new secret'")
    print("3. Name: CIROH_API_KEY")
    print("4. Value: Your API key")
    print("5. Enable notebook access toggle")
    print("6. Run this cell again")
    raise

# === Inputs ===
huc_df    = pd.read_csv("HUC8.csv", dtype=str)
API_URL   = "https://nwm-api.ciroh.org/"
RETURN_EP = f"{API_URL}/return-period"

headers = {"x-api-key": API_KEY}   # ✅ key now comes from Secrets, not hardcoded

# Output directory for all HUCs
per_rp_out_dir = "/content/data/inputs"
os.makedirs(per_rp_out_dir, exist_ok=True)

# === Tunables ===
BATCH_SIZE   = 200
TIMEOUT_S    = 60
MAX_RETRIES  = 3
RETRY_SLEEP  = 2

def fetch_batch(id_batch):
    params = {
        "comids": ",".join(map(str, id_batch)),
        "output_format": "csv",
        "order_by_comid": True
    }
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(RETURN_EP, params=params, headers=headers, timeout=TIMEOUT_S)
            if r.status_code == 200:
                return pd.read_csv(io.StringIO(r.text))
            last_err = f"HTTP {r.status_code}: {r.text[:400]}"
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(RETRY_SLEEP * attempt)
                continue
            break
        except requests.exceptions.RequestException as e:
            last_err = f"Request error: {e}"
            time.sleep(RETRY_SLEEP * attempt)
    raise RuntimeError(f"Batch failed (first 5 IDs {id_batch[:5]} …): {last_err}")

# === Loop over each HUC ===
for huc in huc_df.iloc[:, 0].tolist():
    print(f"\n==================== HUC {huc} ====================")
    feature_ids_path = f"/content/output/flood_{huc}/feature_IDs.csv"
    if not os.path.exists(feature_ids_path):
        print(f"⚠️ Skipping HUC {huc}: {feature_ids_path} not found")
        continue

    raw = pd.read_csv(feature_ids_path)
    col = next((c for c in raw.columns if c.lower().strip() in ("feature_id", "nwm_id", "comid")), None)
    if not col:
        print(f"⚠️ Skipping HUC {huc}: no feature_id column")
        continue

    ids_series = pd.to_numeric(raw[col], errors="coerce").dropna().astype("int64")
    ids_series = ids_series[ids_series > 0]
    ids_unique = ids_series.drop_duplicates().tolist()
    if not ids_unique:
        print(f"⚠️ Skipping HUC {huc}: no valid IDs")
        continue

    order_df = pd.DataFrame({"feature_id": ids_series.tolist(), "_order": range(len(ids_series))})

    # ---- Fetch all batches ----
    frames = []
    for i in range(0, len(ids_unique), BATCH_SIZE):
        batch = ids_unique[i:i + BATCH_SIZE]
        print(f"Fetching {len(batch)} IDs [{i}-{i + len(batch) - 1}] …")
        dfb = fetch_batch(batch)
        if "Unnamed: 0" in dfb.columns:
            dfb = dfb.drop(columns=["Unnamed: 0"])
        frames.append(dfb)

    if not frames:
        print(f"⚠️ No data returned for HUC {huc}")
        continue

    df = pd.concat(frames, ignore_index=True)

    # Normalize column names
    lower   = {c.lower(): c for c in df.columns}
    fid_col = lower.get("feature_id", lower.get("comid"))
    if fid_col != "feature_id":
        df = df.rename(columns={fid_col: "feature_id"})

    # Identify/rename return-period columns
    rp_cols_raw = [c for c in df.columns if "return_period" in c.lower()]
    if not rp_cols_raw:
        print(f"⚠️ No return_period columns found for HUC {huc}")
        continue

    rename_map = {c: f"{c.split('_')[-1]}_year" for c in rp_cols_raw}
    df = df.rename(columns=rename_map)

    # Align with original IDs
    df_one  = df.drop_duplicates(subset=["feature_id"])
    aligned = order_df.merge(
        df_one[["feature_id"] + list(rename_map.values())],
        on="feature_id", how="left"
    ).sort_values("_order").drop(columns=["_order"])

    # ---- Write per-return-period CSVs ----
    written = []
    for rp_col in sorted(rename_map.values(), key=lambda x: int(x.split("_")[0])):
        out_df   = aligned[["feature_id", rp_col]].rename(columns={rp_col: "discharge"})
        rp_label = rp_col.replace("_year", "year")
        out_fp   = os.path.join(per_rp_out_dir, f"{rp_label}_{huc}.csv")
        out_df.to_csv(out_fp, index=False)
        written.append(out_fp)

    print("✅ Wrote files:")
    for p in written:
        print(f"   - {p}")

    # Diagnostics
    rp_cols     = list(rename_map.values())
    missing_ids = aligned[aligned[rp_cols].isna().all(axis=1)]["feature_id"].astype("int64").tolist()
    if missing_ids:
        preview = ", ".join(map(str, missing_ids[:20]))
        more    = f" … (+{len(missing_ids) - 20} more)" if len(missing_ids) > 20 else ""
        print(f"⚠️ {len(missing_ids)} ID(s) had no return-period values: {preview}{more}")
    else:
        print("✅ All feature_ids received at least one return-period value.")

✅ API key loaded securely from Colab Secrets

==================== HUC 10290203 ====================
Fetching 200 IDs [0-199] …
Fetching 200 IDs [200-399] …
Fetching 200 IDs [400-599] …
Fetching 200 IDs [600-799] …
Fetching 200 IDs [800-999] …
Fetching 200 IDs [1000-1199] …
Fetching 200 IDs [1200-1399] …
Fetching 200 IDs [1400-1599] …
Fetching 78 IDs [1600-1677] …
✅ Wrote files:
   - /content/data/inputs/2year_10290203.csv
   - /content/data/inputs/5year_10290203.csv
   - /content/data/inputs/10year_10290203.csv
   - /content/data/inputs/25year_10290203.csv
   - /content/data/inputs/50year_10290203.csv
   - /content/data/inputs/100year_10290203.csv
✅ All feature_ids received at least one return-period value.

==================== HUC 10300102 ====================
Fetching 200 IDs [0-199] …
Fetching 200 IDs [200-399] …
Fetching 200 IDs [400-599] …
Fetching 200 IDs [600-799] …
Fetching 200 IDs [800-999] …
Fetching 200 IDs [1000-1199] …
Fetching 200 IDs [1200-1399] …
Fetching 200 IDs [140

Step 14: Run FIM for all the HUCs

In [ ]:
huc = pd.read_csv('HUC8.csv', dtype= {'HUC8':str})
for i in huc['HUC8']:
  fm.runOWPHANDFIM(i, depth=True)

Completed in 0.57 minutes.

Inundation mapping for 10290203 completed successfully.
Completed in 0.48 minutes.

Inundation mapping for 10290203 completed successfully.
Completed in 0.48 minutes.

Inundation mapping for 10290203 completed successfully.
Completed in 0.5 minutes.

Inundation mapping for 10290203 completed successfully.
Completed in 0.48 minutes.

Inundation mapping for 10290203 completed successfully.
Completed in 0.48 minutes.

Inundation mapping for 10290203 completed successfully.
Completed in 1.7 minutes.

/usr/local/lib/python3.12/dist-packages/numba/core/types/scalars.py:47: RuntimeWarning: invalid value encountered in cast
  return getattr(np, self.name)(value)

Inundation mapping for 10300102 completed successfully.
Completed in 1.82 minutes.

/usr/local/lib/python3.12/dist-packages/numba/core/types/scalars.py:47: RuntimeWarning: invalid value encountered in cast
  return getattr(np, self.name)(value)

Inundation mapping for 10300102 completed successfully.
Comple

Step 15: Merge FIMs for all the HUCs

In [ ]:
import os
import glob
import rasterio
import numpy as np
from collections import defaultdict
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import from_origin

# === INPUTS ===
base_dir = "output"
out_dir  = os.path.join(base_dir, "merged")
os.makedirs(out_dir, exist_ok=True)

# Find all *_depth.tif files
depth_files = glob.glob(os.path.join(base_dir, "flood_*", "*_inundation", "*_depth.tif"))
print(f"📂 Found {len(depth_files)} depth rasters")

# Group rasters by prefix (before first underscore, e.g., "100year")
groups = defaultdict(list)
for fp in depth_files:
    fname = os.path.basename(fp)
    prefix = fname.split("_")[0]
    groups[prefix].append(fp)

# Helper: reproject one raster into union grid
def reproject_to_union(src_path, dst_crs, transform, width, height):
    with rasterio.open(src_path) as src:
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        data = np.empty((src.count, height, width), dtype=src.meta["dtype"])
        for i in range(1, src.count + 1):
            reproject(
                source=rasterio.band(src, i),
                destination=data[i-1],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest
            )
    return data, kwargs

# === MERGE LOOP ===
for prefix, files in groups.items():
    print(f"\n▶ Merging {len(files)} rasters for return period '{prefix}'")

    # 1. Get resolution, CRS, and union bounds
    with rasterio.open(files[0]) as ref:
        res = ref.res
        dst_crs = ref.crs
        union_bounds = ref.bounds

    for f in files[1:]:
        with rasterio.open(f) as src:
            b = src.bounds
            union_bounds = (
                min(union_bounds[0], b.left),
                min(union_bounds[1], b.bottom),
                max(union_bounds[2], b.right),
                max(union_bounds[3], b.top)
            )

    # 2. Compute union transform/grid
    xmin, ymin, xmax, ymax = union_bounds
    xres, yres = res
    width = int((xmax - xmin) / xres)
    height = int((ymax - ymin) / yres)
    transform = from_origin(xmin, ymax, xres, yres)

    # 3. Reproject each raster to union grid
    aligned_data = []
    for f in files:
        data, kwargs = reproject_to_union(f, dst_crs, transform, width, height)
        aligned_data.append(data)

    # 4. Merge (take maximum depth across HUCs)
    merged_data = np.maximum.reduce(aligned_data)

    # 5. Save
    kwargs.update({"compress": "lzw"})
    out_fp = os.path.join(out_dir, f"{prefix}_depth.tif")
    with rasterio.open(out_fp, "w", **kwargs) as dst:
        dst.write(merged_data)

    print(f"✅ Saved merged raster: {out_fp}")

print("\n🎉 All rasters merged successfully.")


📂 Found 18 depth rasters

▶ Merging 3 rasters for return period '2year'
✅ Saved merged raster: output/merged/2year_depth.tif

▶ Merging 3 rasters for return period '25year'
✅ Saved merged raster: output/merged/25year_depth.tif

▶ Merging 3 rasters for return period '5year'
✅ Saved merged raster: output/merged/5year_depth.tif

▶ Merging 3 rasters for return period '100year'
✅ Saved merged raster: output/merged/100year_depth.tif

▶ Merging 3 rasters for return period '10year'
✅ Saved merged raster: output/merged/10year_depth.tif

▶ Merging 3 rasters for return period '50year'
✅ Saved merged raster: output/merged/50year_depth.tif

🎉 All rasters merged successfully.


Step 16: Find the most downstream reach ID

In [ ]:
import geopandas as gpd
import pandas as pd

################## Input Files #########################################
########################################################################
aoi_path = "/content/Multiple_HUC_boundary.shp"
nwm_path = "/content/NWM_streams_EPSG_5070.shp"
########################################################################

aoi     = gpd.read_file(aoi_path)
streams = gpd.read_file(nwm_path)

if streams.crs != aoi.crs:
    aoi = aoi.to_crs(streams.crs)

aoi_union = aoi.geometry.unary_union

# --- Streams intersecting AOI ---
intersects = streams[streams.geometry.intersects(aoi_union)].copy()
intersects["ID"] = intersects["ID"].astype(str)
intersects["to"] = intersects["to"].astype(str)

# --- Boundary crossers: intersects but NOT fully inside ---
crosses = intersects[~intersects.geometry.within(aoi_union)].copy()

# --- True outlet: crossing reach whose 'to' is not another inside reach ---
inside_ids = set(intersects["ID"])
outlet     = crosses[~crosses["to"].isin(inside_ids)]

if outlet.empty:
    print("⚠️ No single outlet found — showing all boundary-crossing reaches:")
    print(crosses[["ID", "to", "order_", "mainstem"]].to_string(index=False))
else:
    row = outlet.iloc[0]
    print(f"✅ AOI Outlet Reach ID : {row['ID']}")
    print(f"   Flows into         : {row['to']}")
    print(f"   Stream order       : {row['order_']}")
    print(f"   Mainstem           : {row['mainstem']}")

    pd.DataFrame([row[["ID", "to", "order_", "mainstem"]]]).to_csv(
        "/content/aoi_outlet.csv", index=False
    )
    print("\n✅ Saved to /content/aoi_outlet.csv")

/tmp/ipykernel_17698/1983795266.py:16: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi_union = aoi.geometry.unary_union


✅ AOI Outlet Reach ID : 6013086.0
   Flows into         : 6013072.0
   Stream order       : 9.0
   Mainstem           : 1.0

✅ Saved to /content/aoi_outlet.csv


In [ ]:
#at the moment this is to be specified by the user
ReachID="6013086" # 👈 The most downstream ReachID

Step 17: Clip the map with area of interest

In [ ]:
# 📚 Imports
import os
import glob
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import pandas as pd

################## Inputs #######################################
aoi_path = "/content/Multiple_HUC_boundary.shp"   # AOI shapefile
unit = "cms"                # Always 'cms' for FIMServe
#################################################################

# 📁 Paths
merged_raster_folder = "/content/output/merged"   # where 2year_depth.tif etc. live
csv_folder = "/content/data/inputs"              # where 2year_xxxx.csv etc. live
output_folder = "/content/final_map"
os.makedirs(output_folder, exist_ok=True)

# Load AOI shapefile and reproject to EPSG:5070
gdf = gpd.read_file(aoi_path)
print(f"📌 Input shapefile CRS: {gdf.crs}")
gdf_5070 = gdf.to_crs(epsg=5070)
print("🔁 Reprojected shapefile to EPSG:5070")

# 🔄 Process each merged *_depth.tif raster
for raster_path in glob.glob(os.path.join(merged_raster_folder, '*_depth.tif')):
    raster_filename = os.path.basename(raster_path)
    print(f"\n🔍 Processing: {raster_filename}")

    try:
        # Get return period prefix (e.g., "2year" from "2year_depth.tif")
        return_period = raster_filename.replace('_depth.tif', '')

        # Search for a CSV that matches this return period
        csv_pattern = os.path.join(csv_folder, f"{return_period}_*.csv")
        csv_files = glob.glob(csv_pattern)

        if not csv_files:
            print(f"⚠️ No CSV files found for return period {return_period}, skipping.")
            continue

        # Look for ReachID in all candidate CSVs
        discharge_value = None
        for csv_path in csv_files:
            df = pd.read_csv(csv_path)
            df = df.astype(str)
            match_row = df[df.apply(lambda row: ReachID in row.values, axis=1)]
            if not match_row.empty:
                discharge_value = int(float(match_row.iloc[0][1]))
                break

        if discharge_value is None:
            print(f"⚠️ ReachID {ReachID} not found in any {return_period} CSVs, skipping.")
            continue

        # Clip raster with AOI
        with rasterio.open(raster_path) as src:
            out_image, out_transform = mask(src, gdf_5070.geometry, crop=True)
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": src.crs
            })

        # Build output filename
        output_name = f"HAND_{ReachID}_{discharge_value}_{unit}_depth.tif"
        output_path = os.path.join(output_folder, output_name)

        # Save
        with rasterio.open(output_path, 'w', **out_meta) as dest:
            dest.write(out_image)

        print(f"✅ Saved clipped raster: {output_path}")

    except Exception as e:
        print(f"❌ Error processing {raster_filename}: {e}")


📌 Input shapefile CRS: EPSG:5070
🔁 Reprojected shapefile to EPSG:5070

🔍 Processing: 50year_depth.tif


/tmp/ipykernel_17698/2042770909.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  discharge_value = int(float(match_row.iloc[0][1]))


✅ Saved clipped raster: /content/final_map/HAND_6013086_20177_cms_depth.tif

🔍 Processing: 100year_depth.tif


/tmp/ipykernel_17698/2042770909.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  discharge_value = int(float(match_row.iloc[0][1]))
/tmp/ipykernel_17698/2042770909.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  discharge_value = int(float(match_row.iloc[0][1]))


✅ Saved clipped raster: /content/final_map/HAND_6013086_22575_cms_depth.tif

🔍 Processing: 2year_depth.tif
✅ Saved clipped raster: /content/final_map/HAND_6013086_8033_cms_depth.tif

🔍 Processing: 25year_depth.tif


/tmp/ipykernel_17698/2042770909.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  discharge_value = int(float(match_row.iloc[0][1]))


✅ Saved clipped raster: /content/final_map/HAND_6013086_17761_cms_depth.tif

🔍 Processing: 10year_depth.tif


/tmp/ipykernel_17698/2042770909.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  discharge_value = int(float(match_row.iloc[0][1]))
/tmp/ipykernel_17698/2042770909.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  discharge_value = int(float(match_row.iloc[0][1]))


✅ Saved clipped raster: /content/final_map/HAND_6013086_14504_cms_depth.tif

🔍 Processing: 5year_depth.tif
✅ Saved clipped raster: /content/final_map/HAND_6013086_11926_cms_depth.tif


Step 18: Download the final flood maps

In [ ]:
# 📦 Zip the folder
!zip -r /content/final_map.zip /content/final_map > /dev/null

# 📥 Provide download link
from google.colab import files
files.download('/content/final_map.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>